# Compute ecDNA score

## Compute and plot the score

In [ ]:
import numpy as np
import seaborn as sns
import os
import pandas as pd
import matplotlib.pyplot as plt
from glob import glob
from datetime import date
from compute_bayes_scores_utils import (
    gather_best_model_results,
    make_01,
)
from scipy.stats import norm
from sklearn.preprocessing import StandardScaler


def setup_dirs(outDir):
    figuresDir = os.path.join(outDir, "figures")
    tablesDir = os.path.join(outDir, "tables")
    dataDir = os.path.join(outDir, "data")
    os.makedirs(dataDir, exist_ok=True)
    os.makedirs(figuresDir, exist_ok=True)
    os.makedirs(tablesDir, exist_ok=True)
    return figuresDir, tablesDir, dataDir


def compute_score(res=None, sd_coeff=0.66):
    """
    Compute ec-DNA score on results from eicicle. Both a normalized and unnormalized score are computed.

    Parameters
    ----------
    res : pd.DataFrame
        DataFrame containing at least the columns 'sigma_alpha_max', 'mu_alpha_max', and 'max_alpha'. Results from running the eicicle.
    sd_coeff : float, optional
        Relative contribution of s.d vs mean in the score computation, by default 0.66

    Returns
    -------
    pd.DataFrame
        Input DataFrame with additional columns for normalized 'sam', 'mam', and computed 'score'
    """
    # Assert all required columns are in res.
    required_cols = ["sigma_alpha_max", "mu_alpha_max", "max_alpha"]
    for col in required_cols:
        assert col in res.columns, f"Column {col} is missing from the input dataframe"
    # Z-score mean and s.d.
    sam = res["sigma_alpha_max"].values.copy()
    sam_unscaled = sam.copy()
    sam = StandardScaler().fit_transform(sam.reshape(-1, 1)).flatten()
    mam = res["mu_alpha_max"].values.copy()
    mam_unscaled = mam.copy()
    mam = StandardScaler().fit_transform(mam.reshape(-1, 1)).flatten()
    res["sam"] = sam
    res["mam"] = mam
    res["sam_unscaled"] = sam_unscaled
    res["mam_unscaled"] = mam_unscaled
    # Compute the ecDNA score
    res["score"] = (res["max_alpha"] ** 2) * (
        sd_coeff * res["sam"] + (1 - sd_coeff) * res["mam"]
    )
    res["score"] = make_01(res["score"])
    res["score_unscaled"] = (res["max_alpha"] ** 2) * (
        sd_coeff * res["sam_unscaled"] + (1 - sd_coeff) * res["mam_unscaled"]
    )
    res["score_unscaled_01"] = make_01(res["score_unscaled"])
    return res


In [ ]:
outDir = "."
figuresDir, tablesDir, dataDir = setup_dirs(outDir)
res_save_path = os.path.join(tablesDir, "best_model_results.csv")
extreme_cases_save_path = os.path.join('../../dat/', "extreme_cases.csv")
NF_OUT_DIR = '.'


### Gather results from the fitted models

In [ ]:
paths = glob.glob(
    f"{NF_OUT_DIR}/**/**/best_model.yaml",
    recursive=True,
)
res = gather_best_model_results(paths)
res.to_csv(res_save_path, index=False)

## Compute the score when multiple samples are available

In [ ]:
# The relative importance of sd vs mean in the final score
sd_coeff = 0.66
res = pd.read_csv(res_save_path)

res = compute_score(res, sd_coeff=sd_coeff)
res.sort_values(by="score", ascending=False, inplace=True)
res.to_csv(res_save_path, index=False)


## Compute ecDNA score along with extreme cases



In [ ]:
sd_coeff = 0.66

# Load extreme cases, expecting columns id , label, sigma_alpha_max, mu_alpha_max, max_alpha
labeled_df = pd.read_csv(extreme_cases_save_path)

# Load the sample scores
res = pd.read_csv(res_save_path)

# Concatenate res
res = pd.concat([res, labeled_df], axis=0, ignore_index=True)

res = compute_score(res, sd_coeff=sd_coeff)
res.sort_values(by="score", ascending=False, inplace=True)

res.to_csv(res_save_path, index=False)


## Compute the score for a single case

We can also report an unnormalized score for a single case. 

In [ ]:
sd_coeff = 0.66
res = pd.read_csv(res_save_path)

res['sam'] = res['sigma_alpha_max'].values.copy()
res['mam'] = res['mu_alpha_max'].values.copy()

# Compute the ecDNA score
res['score'] = (res['max_alpha']**2) * (sd_coeff * res['sam'] + (1-sd_coeff) * res['mam'])

res.to_csv(res_save_path, index=False)